In [7]:
import sys
import os
import warnings
from zoneinfo import ZoneInfo
from datetime import datetime, timezone, timedelta
from pathlib import Path
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import pytz
import json
from pandas import Timestamp
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.arima.model import ARIMA
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score
from scipy.stats import norm

current_dir = Path.cwd()
parent_dir = current_dir.parent
sys.path.insert(0, str(parent_dir))
from lib import *

MODEL_PATH=parent_dir / 'models' 
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 1000)

In [8]:
lookback_minutes = 60
crypto_at = datetime(2026, 5, 4, 4, 30, 0, tzinfo=ZoneInfo('America/Chicago'))
series = 'KXBTC15M'
ticker = 'KXBTC15M-26MAY040515-15'
series, event_dt = parse_kalshi_15m_event_ticker(ticker)
df_api = get_market_data_from_api(series, crypto_at, lookback_minutes)
df_api = df_api.set_index('datetime')
df_api.index = df_api.index.tz_convert('America/Chicago')
df_api.sort_index(inplace=True)
print(f"df_api: {len(df_api)}")
df_api

df_api: 60


,ticker,volume_fp,open_interest_fp,floor_strike,end_period_ts,yes_ask_open_dollar,yes_ask_high_dollar,yes_ask_low_dollar,yes_ask_close_dollar,yes_bid_open_dollar,yes_bid_high_dollar,yes_bid_low_dollar,yes_bid_close_dollar,no_ask_open_dollar,no_ask_high_dollar,no_ask_low_dollar,no_ask_close_dollar,no_bid_open_dollar,no_bid_high_dollar,no_bid_low_dollar,no_bid_close_dollar
datetime,,,,,,,,,,,,,,,,,,,,,
2026-05-04 03:16:00-05:00,KXBTC15M-26MAY040430-30,10036.16,7219.16,79772.80,1777882560,0.999,0.999,0.490,0.510,0.000,0.590,0.000,0.500,1.000,0.410,1.000,0.500,0.001,0.001,0.510,0.490
2026-05-04 03:17:00-05:00,KXBTC15M-26MAY040430-30,15070.96,13553.00,79772.80,1777882620,0.510,0.510,0.280,0.290,0.500,0.500,0.270,0.270,0.500,0.500,0.730,0.730,0.490,0.490,0.720,0.710
2026-05-04 03:18:00-05:00,KXBTC15M-26MAY040430-30,6651.52,16715.65,79772.80,1777882680,0.290,0.300,0.230,0.250,0.270,0.290,0.210,0.240,0.730,0.710,0.790,0.760,0.710,0.700,0.770,0.750
2026-05-04 03:19:00-05:00,KXBTC15M-26MAY040430-30,14076.34,21052.40,79772.80,1777882740,0.250,0.440,0.250,0.390,0.240,0.430,0.240,0.380,0.760,0.570,0.760,0.620,0.750,0.560,0.750,0.610
2026-05-04 03:20:00-05:00,KXBTC15M-26MAY040430-30,20702.41,23347.74,79772.80,1777882800,0.390,0.500,0.350,0.450,0.380,0.490,0.330,0.440,0.620,0.510,0.670,0.560,0.610,0.500,0.650,0.550
2026-05-04 03:21:00-05:00,KXBTC15M-26MAY040430-30,16702.91,24413.49,79772.80,1777882860,0.450,0.500,0.300,0.330,0.440,0.490,0.290,0.320,0.560,0.510,0.710,0.680,0.550,0.500,0.700,0.670
2026-05-04 03:22:00-05:00,KXBTC15M-26MAY040430-30,8835.01,24075.70,79772.80,1777882920,0.330,0.410,0.220,0.220,0.320,0.400,0.190,0.210,0.680,0.600,0.810,0.790,0.670,0.590,0.780,0.780
2026-05-04 03:23:00-05:00,KXBTC15M-26MAY040430-30,15701.50,29679.74,79772.80,1777882980,0.220,0.220,0.048,0.058,0.210,0.210,0.043,0.052,0.790,0.790,0.957,0.948,0.780,0.780,0.952,0.942
2026-05-04 03:24:00-05:00,KXBTC15M-26MAY040430-30,23026.13,38171.43,79772.80,1777883040,0.058,0.070,0.018,0.028,0.052,0.056,0.014,0.019,0.948,0.944,0.986,0.981,0.942,0.930,0.982,0.972


In [9]:
df_crypto = get_crypto_past_minutes(series, crypto_at, lookback_minutes)
df_crypto['datetime'] = pd.to_datetime(df_crypto['datetime'])
df_crypto['datetime'] = df_crypto['datetime'].dt.tz_convert('America/Chicago')
df_crypto['datetime'] = df_crypto['datetime'].dt.floor('min')
df_crypto = df_crypto.set_index('datetime')
df_crypto

,open,high,low,close,tick_count
datetime,,,,,
2026-05-04 03:31:00-05:00,79729.82,79738.31,79719.64,79720.00,34
2026-05-04 03:32:00-05:00,79722.16,79722.16,79719.70,79719.70,0
2026-05-04 03:33:00-05:00,79744.36,79765.66,79734.89,79742.61,53
2026-05-04 03:34:00-05:00,79739.24,79739.90,79734.67,79739.49,13
2026-05-04 03:35:00-05:00,79728.05,79728.05,79712.82,79712.82,36
2026-05-04 03:36:00-05:00,79680.74,79687.42,79680.74,79687.42,0
2026-05-04 03:37:00-05:00,79729.01,79730.29,79694.30,79697.07,40
2026-05-04 03:38:00-05:00,79699.49,79699.49,79697.64,79697.64,0
2026-05-04 03:39:00-05:00,79696.43,79696.43,79695.66,79695.66,0


In [ ]:
print(f"df_crypto: {len(df_crypto)}")
filter_timestamp = df_crypto[df_crypto.index.minute.isin([0,15,30,45])].index[0]
df_crypto = df_crypto[df_crypto.index >= filter_timestamp]
df_merged = df_crypto.join(df_api, how='left')
df_merged['yes_dist'] = df_merged['close'] - df_merged['floor_strike']
df_merged['log_return'] = np.log(df_merged['close'] / df_merged['close'].shift(1))
df_merged['3m_log_return'] = df_merged['log_return'].rolling(3).std()
df_merged['5m_log_return'] = df_merged['log_return'].rolling(5).std()
df_merged['ma3'] = df_merged['close'].rolling(3).mean()
df_merged['ma5'] = df_merged['close'].rolling(5).mean()
df_merged['ma3_vs_strike'] = (df_merged['ma3'] - df_merged['floor_strike'])/df_merged['floor_strike'] * 100
df_merged['ma5_vs_strike'] = (df_merged['ma5'] - df_merged['floor_strike'])/df_merged['floor_strike'] * 100
df_merged['yes_dist_pct'] = df_merged['yes_dist'] / df_merged['floor_strike'] * 100
df_merged['1m_yes_dist_momentum'] = df_merged['yes_dist'] - df_merged['yes_dist'].shift(1)
df_merged['3m_yes_dist_momentum'] = df_merged['yes_dist'] - df_merged['yes_dist'].shift(3)
df_merged['5m_yes_dist_momentum'] = df_merged['yes_dist'] - df_merged['yes_dist'].shift(5)
df_merged['time_decay'] = np.where(df_merged.index.minute % 15 == 0, 0, 15 - df_merged.index.minute % 15)
# df_merged = df_merged.dropna()
df_merged